<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/atlassian_git_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
import os

def scrape_page(url):
    """
    Fetches a webpage, extracts headings, paragraphs, and code blocks,
    and returns the extracted text and a list of internal links.
    """
    print(f"Scraping: {url}")
    content = []
    links = set()
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title if available
        if soup.title and soup.title.string:
            content.append(f"# {soup.title.string.strip()}\n")

        # Extract headings, paragraphs, and code blocks
        for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'pre', 'code']):
            if tag.name.startswith('h'):
                content.append(f"\n{tag.name[0]} {tag.get_text(strip=True)}\n")
            elif tag.name == 'p':
                text = tag.get_text(strip=True)
                if text:
                    content.append(f"{text}\n")
            elif tag.name == 'pre':
                # For 'pre' tags, try to get code content specifically
                code_content = tag.get_text(strip=True)
                if code_content:
                    content.append(f"\n```\n{code_content}\n```\n")
            elif tag.name == 'code':
                # If 'code' is not inside 'pre', wrap it in backticks
                if not tag.find_parent('pre'):
                    code_content = tag.get_text(strip=True)
                    if code_content:
                        content.append(f"`{code_content}` ") # Add space to separate inline code

        # Find all internal links
        base_url_parsed = urlparse(url)
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            full_url = urljoin(url, href)
            parsed_full_url = urlparse(full_url)

            # Only consider links within the same domain and not anchors or mailto
            if parsed_full_url.netloc == base_url_parsed.netloc and \
               not full_url.startswith('mailto:') and \
               not parsed_full_url.fragment:
                links.add(full_url)

    except requests.exceptions.RequestException as e:
        print(f"Error scraping {url}: {e}")
    return "\n".join(content), links


In [ ]:
def scrape_website(start_url, max_pages=50):
    """
    Scrapes a website starting from `start_url`, following internal links.
    """
    base_domain = urlparse(start_url).netloc
    visited_urls = set()
    urls_to_visit = [start_url]
    all_collected_content = []
    page_count = 0

    while urls_to_visit and page_count < max_pages:
        current_url = urls_to_visit.pop(0)
        if current_url in visited_urls:
            continue

        page_content, new_links = scrape_page(current_url)
        if page_content:
            all_collected_content.append(f"\n\n---\n\n## Content from: {current_url}\n\n{page_content}")
            visited_urls.add(current_url)
            page_count += 1

            for link in new_links:
                parsed_link = urlparse(link)
                # Only add links that are within the same domain and not yet visited
                if parsed_link.netloc == base_domain and link not in visited_urls and link not in urls_to_visit:
                    urls_to_visit.append(link)

    return "\n".join(all_collected_content)


In [ ]:
start_url = 'https://www.atlassian.com/git/tutorials'
scraped_data = scrape_website(start_url)

output_filename = 'atlassian_git_tutorials.md'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(scraped_data)

print(f"Scraping complete. Content saved to {output_filename}")
print("You can download the file from the Colab file browser (left sidebar -> folder icon).")


Scraping: https://www.atlassian.com/git/tutorials
Scraping: https://www.atlassian.com/git/tutorials/why-git
Scraping: https://www.atlassian.com/company/careers
Scraping: https://www.atlassian.com/software/startups
Scraping: https://www.atlassian.com/software/customer-service-management
Scraping: https://www.atlassian.com/company
Scraping: https://www.atlassian.com/work-management/project-collaboration
Scraping: https://www.atlassian.com/git/tutorials/using-branches/merge-strategy
Scraping: https://www.atlassian.com/git/tutorials/merging-vs-rebasing
Scraping: https://www.atlassian.com/git/tutorials/undoing-changes/git-clean
Scraping: https://www.atlassian.com/software/confluence
Scraping: https://www.atlassian.com/software/bitbucket
Scraping: https://www.atlassian.com/resources
Scraping: https://www.atlassian.com/software/rovo-dev
Scraping: https://www.atlassian.com/git/tutorials/git-bash
Scraping: https://www.atlassian.com/software/small-business
Scraping: https://www.atlassian.com/leg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive_output_dir = '/content/drive/MyDrive/rag_git/git_scraper_results'
os.makedirs(drive_output_dir, exist_ok=True)

drive_output_path = os.path.join(drive_output_dir, output_filename)

# Copy the file to Google Drive
import shutil
shutil.copy(output_filename, drive_output_path)

print(f"File '{output_filename}' successfully copied to Google Drive at '{drive_output_path}'")


File 'atlassian_git_tutorials.md' successfully copied to Google Drive at '/content/drive/MyDrive/rag_git/git_scraper_results/atlassian_git_tutorials.md'
